# AEGIS-SQL — Spider-KO Schema Representation Ablation

Qwen2.5-Coder-1.5B-Instruct + NF4 4-bit을 고정하고 `slm / ddl / compact / mschema` 네 가지 스키마 표현만 바꿔 Spider-KO validation 1,034문항을 비교합니다.

중단되면 같은 Drive 경로에서 다시 `런타임 → 모두 실행`하면 완료된 스타일은 건너뜁니다. 기존 실험의 Git/runtime provenance와 현재 환경이 다르면 runner가 비교를 거부합니다.

> 결과는 **공식 Spider leaderboard 점수**가 아니라 AEGIS `execution_match` 기반 외부 cross-domain 일반화 실험입니다.


In [ ]:
import subprocess
import torch

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("GPU 런타임이 필요합니다. Colab 런타임 설정에서 T4/L4 GPU를 선택하세요.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ARCHIVE_DIR = Path("/content/drive/MyDrive/AEGIS-SQL")
RUN_DIR = ARCHIVE_DIR / "spider-ko-schema-ablation"
RUN_META = RUN_DIR / "run-meta.json"
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
print("persistent archive:", RUN_DIR)


In [ ]:
import json
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/aegis-sql")
REPO_URL = "https://github.com/sokldjs554/aegis-sql.git"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", "main"], cwd=REPO_DIR, check=True)

if RUN_META.exists():
    pinned_sha = json.loads(RUN_META.read_text(encoding="utf-8"))["git_sha"]
    subprocess.run(["git", "fetch", "origin", pinned_sha], cwd=REPO_DIR, check=False)
    subprocess.run(["git", "checkout", "--detach", pinned_sha], cwd=REPO_DIR, check=True)
    print("resume pinned git SHA:", pinned_sha)
else:
    subprocess.run(["git", "checkout", "--detach", "origin/main"], cwd=REPO_DIR, check=True)
    print("new ablation git SHA:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())


In [ ]:
import os
import subprocess

env = os.environ.copy()
env["ARCHIVE_DIR"] = str(ARCHIVE_DIR)
subprocess.run(
    "bash scripts/run_spider_ko_schema_ablation_colab.sh",
    cwd=REPO_DIR,
    shell=True,
    check=True,
    env=env,
)


In [ ]:
import json

SUMMARY = RUN_DIR / "spider-ko-schema-ablation-summary.json"
summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("comparable:", summary["comparable"])
print("winner:", summary["winner"])
print()
print(f"{'style':<9} {'EX':>8} {'dEX(pp)':>9} {'exec_fail':>10} {'schema_ref':>11} {'column':>8} {'table':>7} {'p50':>9} {'p95':>9}")
for style in ("slm", "ddl", "compact", "mschema"):
    m = summary["styles"][style]
    latency = m["latency_ms"]
    print(
        f"{style:<9} {m['execution_accuracy']:>8.1%} {m['delta_accuracy_pp_vs_slm']:>9.2f} "
        f"{m['execution_failures']:>10} {m['schema_reference_failures']:>11} "
        f"{m['no_such_column']:>8} {m['no_such_table']:>7} "
        f"{latency.get('p50', 0):>9.2f} {latency.get('p95', 0):>9.2f}"
    )


## 해석

1. 우선 EX 변화와 `schema_reference_failures` (`no such column` + `no such table`) 감소를 함께 봅니다.
2. 가장 좋은 schema style을 고른 뒤에만 execution-guided repair를 실험합니다.
3. 그 이후에도 nested/set-op reasoning이 낮게 남을 때 3B 모델 비교로 넘어갑니다.
4. KorFin-Bench와 이 결과를 하나의 점수로 합치지 않습니다. 이 수치는 공식 Spider leaderboard 점수가 아닙니다.
